Here we will be using a pdf file which contains the information about the Series of Llama models. The pdf file is accumulated from the wikipedia page of Llama. The pdf file is available in the GitHub repository of this blog.

In [2]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

def create_chunks_from_pdf(data_path, chunk_size, chunk_overlap):

   '''
   This function takes a directory of PDF files and creates chunks of text from each file.
   The text is split into chunks of size `chunk_size` with an overlap of `chunk_overlap`.
   This chunk is then converted into a langchain Document object.

   Args:
      data_path (str): The path to the directory containing the PDF files.
      chunk_size (int): The size of each chunk.
      chunk_overlap (int): The overlap between each chunk.

   Returns:
      docs (list): A list of langchain Document objects, each containing a chunk of text.
   '''

   # Load the documents from the directory
   loader = DirectoryLoader(data_path, loader_cls=PyPDFLoader)

   # Split the documents into chunks
   text_splitter = RecursiveCharacterTextSplitter(
      chunk_size=chunk_size,
      chunk_overlap=chunk_overlap,
      length_function=len,
      is_separator_regex=False,
   )
   docs = loader.load_and_split(text_splitter=text_splitter)
   return docs

data_path = r'C:\Users\Tifa\Desktop\rag\data'
chunk_size = 500
chunk_overlap = 50

docs = create_chunks_from_pdf(data_path, chunk_size, chunk_overlap)


[Document(metadata={'source': 'C:\\Users\\Tifa\\Desktop\\rag\\data\\circulaire.pdf', 'page': 0}, page_content="CIRCULAIRE AUX INT ERMEDIAIRES \nAGR EES N°87-37 D U 24 SEPTE MBR E 1987   ∗ \nOBJET : Comptes spéciaux en  devises et en dina rs \nconv ertibles.  \nL'article  25 nouveau du décret  n°77 -608 du 27 \njuillet  1977 fixant  les modalités  d'application du code \ndes changes di spense  de l'obligation de cess ion l es \ndevises provenant des  revenus ou produits des avoirs à \nl'étranger  et des  avoi rs en devises à l'étranger  déclarés \nà la Banque Centrale  de Tunisie confor mém ent aux"),
 Document(metadata={'source': 'C:\\Users\\Tifa\\Desktop\\rag\\data\\circulaire.pdf', 'page': 0}, page_content="articles 16 et 18 du code des changes et à l'article 16 de \nla loi n°86 -83 du 1er septembre 1986 port ant loi de \nfinances recti ficative pour l'année 1986. \nCes devises peuvent être logées dans des co mptes \nspéciaux  en  dev ises  ou  en  dinars  conver tibles  et \npeuvent

We first start by loading the documents from the directory. We then split the documents into several chunks of equal size and finally convert them to langchain docs

Now the next step is to index the documents in the Qdrant database. But before moving on to that step, we will first need to load the embeddings model which we will be using to convert the documents into embeddings. To build the simple RAG model we will first start with 'BAAI/bge-large-en' embeddings model. In the evaluation section, we will also try with other embeddings models to see how the performance changes.

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_models = ['BAAI/bge-large-en']

# Load the embeddings model
embeddings = HuggingFaceEmbeddings(model_name=embedding_models[0])

c:\Users\Tifa\Desktop\rag\.venv\lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange
Error while downloading from https://cdn-lfs.huggingface.co/repos/60/5e/605eb2707e17d287d9db515a55d5abd41f99516f676822cdf999ae87d847c1a2/37136ad03a0da3ea220bc31850c5b49f39d56fa0d99ebd48887d0c9bb60ad5d1?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27model.safetensors%3B+filename%3D%22model.safetensors%22%3B&Expires=1723407157&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTcyMzQwNzE1N319LCJSZXNvdXJjZSI6Imh0dHBzOi8vY2RuLWxmcy5odWdnaW5nZmFjZS5jby9yZXBvcy82MC81ZS82MDVlYjI3MDdlMTdkMjg3ZDlkYjUxNWE1NWQ1YWJkNDFmOTk1MTZmNjc2ODIyY2RmOTk5YWU4N2Q4NDdjMWEyLzM3MTM2YWQwM2EwZGEzZWEyMjBiYzMxODUwYzViNDlmMzlkNTZmYTBkOTllYmQ0ODg4N2QwYzliYjYwYWQ1ZDE%7EcmVzcG9uc2UtY29u

ChunkedEncodingError: ('Connection broken: IncompleteRead(32768 bytes read, 417836968 more expected)', IncompleteRead(32768 bytes read, 417836968 more expected))

In [6]:
from langchain_community.embeddings.ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="mistral")


Great, Now that we have created the chunked documents, loaded the embeddings model, we can now index the documents using Qdrant.

In [7]:
from langchain_qdrant import Qdrant

def index_documents_and_retrieve(docs, embeddings):

    '''
    This function uses the Qdrant library to index the documents using the chunked text and embeddings model.
    For the simplicity of the example, we are using in-memory storage only.

    Args:
    docs: List of documents generated from the document loader of langchain
    embeddings: List of embeddings generated from the embeddings model

    Returns:
    retriever: Qdrant retriever object which can be used to retrieve the relevant documents
    '''

    qdrant = Qdrant.from_documents(
        docs,
        embeddings,
        location=":memory:",  # Local mode with in-memory storage only
        collection_name="my_documents",
    )

    retriever = qdrant.as_retriever()

    return retriever

retriever = index_documents_and_retrieve(docs, embeddings)

In [8]:
retriever

VectorStoreRetriever(tags=['Qdrant', 'OllamaEmbeddings'], vectorstore=<langchain_qdrant.vectorstores.Qdrant object at 0x000001F8B7E3EB00>)

In [12]:
questions = [
    "Quelle est la date de la circulaire mentionnée?",
    "Quelles sont les nouvelles directives pour la mise en œuvre?",
]

ground_truth = [
    "La circulaire est datée du 5 août 2023.",
    "Les nouvelles directives stipulent que toutes les unités doivent suivre les procédures mises à jour.",
]


For the simplicity of this blog, we have indexed the documents in the Qdrant database using the in-memory mode. But in the production environment, you can use the persistent mode to store the indexed documents in the disk. If you want to know more about these approaches you can visit my other blogs on Qdrant.

Now that we have the retriever object, we can use it to retrieve the relevant documents based on the query and finally generate the answer using the LLM model.

In [9]:
from langchain_community.llms.ollama import Ollama

# Load the Llama-3 model using the Ollama
llm = Ollama(model="mistral")

Now let's build the simple RAG chain from the retriever and the LLM model.

In [10]:
from langchain_core.prompts import PromptTemplate
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser

def build_rag_chain(llm, retriever):

    '''
    This function builds the RAG chain using the LLM model and the retriever object. 
    The RAG chain is built using the following steps:
    1. Retrieve the relevant documents using the retriever object
    2. Pass the retrieved documents to the LLM model along with prompt generated using the context and question
    3. Parse the output of the LLM model

    Args:
    llm: LLM model object
    retriever: Qdrant retriever object

    Returns:
    rag_chain: RAG chain object which can be used to answer the questions based on the context
    '''
    
    template = """
        Answer the question based only on the following context:
        
        {context}
        
        Question: {question}
        """
    
    prompt = PromptTemplate(
        template=template,
        input_variables=["context","question"]
        )
    
    rag_chain = (
        {"context": retriever,  "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    
    return rag_chain

rag_chain = build_rag_chain(llm, retriever)

Now that we have the RAG model, we can use it to generate the answers for the queries. Let's test it out

In [11]:
rag_chain.invoke('What is this document about?')

' The provided documents seem to be about regulations regarding accounts (specifically special accounts) in Tunisia, focusing on accounts for foreign nationals residing in Tunisia and the procedures for opening, managing, and declaring these accounts. The accounts can be in foreign currencies or Tunisian dinars convertible. The documents also mention conditions related to interest produced by deposits and reporting of revenues or assets from abroad.'

There you go! You have successfully built a RAG App using Langchain, Qdrant, HuggingFace, Ollama and Llama-3. You can now use this RAG App to answer questions based on the context of the documents.

Now let's move on to the next section where we will learn how to evaluate the RAG App using the evaluation metrics discussed in the above sections.

In [13]:
from datasets import Dataset

def create_test_case(questions, ground_truth, rag_chain, retriever):
    '''
    This function creates a test case for the RAG model
    It takes a list of questions and the corresponding ground truth answers. 
    It then uses the RAG model to generate answers for the questions.
    It also retrieves the relevant documents for each question.
    Finally, it combines all the information into a dataset object and returns it.

    Args:
        questions: list of strings, questions to be answered
        ground_truth: list of strings, corresponding ground truth answers
        rag_chain: RAG model
        retriever: Retriever object

    Returns:
        dataset: Dataset object containing the questions, answers, contexts and ground truth answers
    '''
    
    data = {"question": [], "answer": [], "contexts": [], "ground_truth": ground_truth}

    for query in questions:
        data["question"].append(query)
        # data["answer"].append(rag_chain.invoke(query)['result'])
        data["answer"].append(rag_chain.invoke(query))
        data["contexts"].append([doc.page_content for doc in retriever.get_relevant_documents(query)])

    dataset = Dataset.from_dict(data)

    return dataset

In [14]:
from ragas import evaluate

from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_recall,
    context_precision,
)

import nest_asyncio
nest_asyncio.apply()

def evaluate_rag(dataset, llm, embeddings):

    '''
    This function evaluates the RAG model on a dataset using the specified metrics

    Args:
        dataset: Dataset object containing the questions, answers, contexts and ground truth answers
        llm: LLM model
        embeddings: Embeddings object

    Returns:
        result: dictionary containing the evaluation results
    '''
    result = evaluate(
        dataset=dataset,
        llm=llm,
        embeddings=embeddings,
        metrics=[
            context_precision,
            faithfulness,
            answer_relevancy,
            context_recall,
        ],
        raise_exceptions=True
    )

    return result

In [17]:
import sys
#sys.path.append('..')

from tqdm.notebook import tqdm

# load the utility functions


# empty lists to store the results
context_relevancy_result, context_precision_result, context_recall_result, faithfulness_result, answer_relevancy_result = [], [], [], [], []

docs = create_chunks_from_pdf(data_path, chunk_size, chunk_overlap) # ceate langchain chunked documents
retriever = index_documents_and_retrieve(docs, embeddings) # index the documents and get the retriever object
rag_chain = build_rag_chain(llm, retriever) # build the RAG model
dataset = create_test_case(questions, ground_truth, rag_chain, retriever) # create the test case for each question using each embedding model

    # store the results


In [18]:
evaluation_result = evaluate_rag(dataset, llm, embeddings) # evaluate the RAG model on the test case

Evaluating:   0%|          | 0/8 [03:00<?, ?it/s]


TimeoutError: 

In [19]:
context_relevancy_result.append(evaluation_result['context_relevancy'])
context_precision_result.append(evaluation_result['context_precision'])
context_recall_result.append(evaluation_result['context_recall'])
faithfulness_result.append(evaluation_result['faithfulness'])
answer_relevancy_result.append(evaluation_result['answer_relevancy'])

    # print the results to see how the model is performing
print(f"Embedding Model:")
for rows in evaluation_result.to_pandas().iterrows():
    print(f"Question: {rows[1]['question']}")
    print(f"Answer: {rows[1]['answer']}")
    print("Ground Truth: ", rows[1]['ground_truth'])
    print(f"Context Relevancy: {rows[1]['context_relevancy']}")
    print(f"Context Precision: {rows[1]['context_precision']}")
    print(f"Faithfulness: {rows[1]['faithfulness']}")
    print(f"Answer Relevancy: {rows[1]['answer_relevancy']}")
    print(f"Context Recall: {rows[1]['context_recall']}")
    print("="*100)
print("\\_/"*50)

NameError: name 'evaluation_result' is not defined